# Anonymize the customer-complaints example with `itk-anonymize`

This notebook uses **`packages/itk-anonymize`** (`import itk_anonymize`), not the legacy
`intelligence_toolkit.anonymize_case_data` module or the Streamlit workflow. It reads one
existing repository input directly, without copying it or generating new data:

- `example_outputs/anonymize_case_data/customer_complaints/customer_complaints_prepared.csv`

That file has 3,000 customer-complaint records with nine attributes: `city`, `age_range`,
five `*_issue` flags, `product_code` and `period`. The notebook prepares the records,
checks how synthesizable they are, releases differentially private (DP) aggregates and
synthetic records through pacsynth, measures the error that privacy introduced, and
answers the same questions the workflow's charts answer (top attributes, time series,
flows between attributes) from the released data only.

Differential privacy adds fresh random noise on every run and pacsynth exposes no seed,
so counts, error percentages and the number of synthetic records differ a little each
time this notebook runs. The sensitive data never leaves this process: everything shown
after step 3 is computed from `result`, which holds only the protected outputs and the
parsed aggregates needed to query them.

## 1. Install and select a Python kernel

Use **Python 3.11 or 3.12**. The package itself only needs `polars` and `pydantic`; the
DP engine is `pac-synth`, which the repository already pins. From the repository root, run
once in a terminal:

```powershell
uv venv --python 3.12 .venv-anonymize-demo
uv pip install --python .venv-anonymize-demo\Scripts\python.exe -e .\packages\itk-anonymize pac-synth==0.0.8 plotly ipykernel
.\.venv-anonymize-demo\Scripts\python.exe -m ipykernel install --user --name itk-anonymize-demo --display-name "Python (itk-anonymize demo)"
```

On macOS/Linux use `.venv-anonymize-demo/bin/python` and slash-separated paths. If you
already use the repository's own environment (`uv sync --extra dev`), its kernel works
too, since the workspace installs `itk-anonymize` in development mode.

Open this notebook in VS Code or Jupyter, select the kernel, and run all cells. Input
lookup walks up from the working directory, so the notebook runs from the repository root
or from its own folder. `plotly` is optional: the chart cells skip themselves when it is
not installed, and every chart is drawn from a frame shown just above it.

In [ ]:
# Copyright (c) 2026 Microsoft Corporation. All rights reserved.
# Licensed under the MIT license. See LICENSE file in the project.

from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import polars as pl
from IPython.display import Markdown
from IPython.display import display as show
from itk_anonymize import Anonymizer, AnonymizeConfig, AttributeValue
from itk_anonymize.prepare import prepare_frame, suppress_boolean_binary
from itk_anonymize.queries import (
    aggregate_links,
    synthetic_links,
    time_series,
    top_attributes,
)
from itk_anonymize.selection import data_schema, describe_selection

try:
    import plotly.express as px
except ImportError:  # charts are optional
    px = None

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(60)

cwd = Path.cwd().resolve()
relative_input = (
    Path("example_outputs")
    / "anonymize_case_data"
    / "customer_complaints"
    / "customer_complaints_prepared.csv"
)
repo_root = next(
    (parent for parent in (cwd, *cwd.parents) if (parent / relative_input).is_file()),
    None,
)
if repo_root is None:
    message = "Run from within the repository containing the customer-complaints input."
    raise FileNotFoundError(message)
input_path = repo_root / relative_input
print("Input:", input_path.relative_to(repo_root))


def installed_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:  # dev-mode checkout without distribution metadata
        return "dev"


packages = ["itk-anonymize", "pac-synth", "polars", "pydantic"]
show(
    pl.DataFrame(
        {
            "package": packages,
            "version": [installed_version(name) for name in packages],
        }
    )
)

## 2. Prepare the sensitive records

`prepare_frame` turns every column into text with `""` for missing values, which is the
form DP engines consume (whole-number floats lose their `.0`). `suppress_boolean_binary`
then blanks the `False` side of the five issue flags: the release only needs to know
which issues were *present* in a complaint, and dropping the absent side halves the
attribute space those flags occupy.

Everything is read as text so that `age_range` buckets such as `(30-40]` and `period`
labels such as `2023-H1` survive unchanged. The schema table lists the distinct values
each attribute can take; these are the terms the selections in step 5 are built from.

In [ ]:
raw = pl.read_csv(input_path, infer_schema_length=0)
sensitive = suppress_boolean_binary(prepare_frame(raw))
assert sensitive.height == raw.height
assert sensitive.columns == raw.columns

show(Markdown(f"**First 10 of {sensitive.height} prepared rows**"))
show(sensitive.head(10))

schema = data_schema(sensitive)
show(Markdown("**Attribute values (the `False` side of each issue flag is blank)**"))
show(
    pl.DataFrame(
        {
            "attribute": list(schema),
            "distinct_values": [len(values) for values in schema.values()],
            "values": [", ".join(values) for values in schema.values()],
        }
    )
)

## 3. How synthesizable is the data?

Synthesizability compares the attribute-combination space with the number of records.
`possible_combinations` is the product of the per-column value counts (plus the blank),
`mean_vals_per_record` is how many non-blank attributes a typical record carries, and
`max_combinations_per_record` is `2 ** mean_vals_per_record`. The
`excess_combinations_ratio` divides the second by the record count: well below 1 means
most combinations appear many times and can be released with little noise, well above 1
means many combinations are unique to one record and will be suppressed or fabricated.

In [ ]:
stats = Anonymizer.analyze_synthesizability(sensitive)
show(
    pl.DataFrame(
        {
            "statistic": [
                "num_cols",
                "overall_att_count",
                "possible_combinations",
                "possible_combinations_per_row",
                "mean_vals_per_record",
                "max_combinations_per_record",
                "excess_combinations_ratio",
            ],
            "value": [
                stats.num_cols,
                stats.overall_att_count,
                stats.possible_combinations,
                stats.possible_combinations_per_row,
                stats.mean_vals_per_record,
                stats.max_combinations_per_record,
                stats.excess_combinations_ratio,
            ],
        }
    )
)

## 4. Release DP aggregates and synthetic records

`AnonymizeConfig` holds the privacy budget and the pacsynth settings. `epsilon=12.0` is the
value the workflow defaults to for this example; smaller values give stronger privacy and
larger errors. `reporting_length=4` means every combination of up to four attribute
values is counted. `Anonymizer.from_pacsynth()` is the only place pacsynth is touched; any
object with a `synthesize(table, config)` method can stand in for it, which is how the
package's tests avoid running DP at all.

The result carries three aggregate dictionaries keyed by sorted attribute-value tuples:
the exact sensitive counts (kept only for the error report below and never shown),
the DP-protected counts, and the counts recomputed from the synthetic records. `delta`
is the standard `1 / (n ln n)` for `n` protected records.

In [ ]:
config = AnonymizeConfig(epsilon=12.0, reporting_length=4)
anonymizer = Anonymizer.from_pacsynth()
result = anonymizer.anonymize(sensitive, config)

show(
    pl.DataFrame(
        {
            "quantity": [
                "sensitive records",
                "protected record count (DP)",
                "synthetic records sampled",
                "delta",
                "sensitive combinations counted",
                "DP aggregates released",
                "combinations in synthetic data",
            ],
            "value": [
                f"{sensitive.height:,}",
                f"{result.protected_record_count:,}",
                f"{result.synthetic.height:,}",
                f"{result.delta:.3e}",
                f"{len(result.sensitive_aggregates):,}",
                f"{len(result.dp_aggregates):,}",
                f"{len(result.synthetic_aggregates):,}",
            ],
        }
    )
)
show(Markdown(f"**First 10 of {result.synthetic.height} synthetic records**"))
show(result.synthetic.head(10))

The aggregate frame uses the Synthetic Data Showcase file layout: one `selections` row per
attribute combination (parts joined by `;`) with its `protected_count`, plus a
`record_count` row holding the DP number of records. It sorts by count and then by key, so
two runs with the same counts print in the same order. Both frames can be written out with
`write_csv` to share the release without the sensitive input.

In [ ]:
aggregates = result.aggregates_frame
show(Markdown(f"**Largest 15 of {aggregates.height} DP aggregates**"))
show(aggregates.head(15))
show(
    Markdown(
        "**Released aggregates by combination length**"
        " (how many combinations of each length survived DP)"
    )
)
show(
    pl.DataFrame({"length": [len(key) for key in result.dp_aggregates]})
    .group_by("length")
    .agg(pl.len().alias("combinations"))
    .sort("length")
)

### Error reports: what did privacy cost?

`Anonymizer.evaluate` compares each released aggregate set with the exact sensitive
counts, per combination length and overall:

- **mean_count**: average exact count of the combinations that were released.
- **mean_error**: average absolute difference between exact and released counts.
- **suppressed_pct**: share of exact combinations that received no released count.
- **fabricated_pct**: share of released combinations that do not exist in the sensitive data.

The first report scores the DP aggregates, the second the synthetic records. Short
combinations are usually near exact; long ones carry most of the suppression and noise.
`to_text_frame()` renders the same numbers the way the Streamlit workflow does.

In [ ]:
anonymizer.evaluate(result)
assert result.aggregate_error_report is not None
assert result.synthetic_error_report is not None

show(Markdown("**DP aggregates vs sensitive counts**"))
show(result.aggregate_error_report.frame)
show(Markdown("**Synthetic records vs sensitive counts**"))
show(result.synthetic_error_report.frame)
show(Markdown("**Same aggregate report, legacy text layout**"))
show(result.aggregate_error_report.to_text_frame())

## 5. Query the released data

The query functions take the synthetic frame and the DP aggregates and return frames with
the same columns the workflow's charts are drawn from. Where a combination was counted
within the reporting length they report the exact DP count; otherwise they fall back to
counting synthetic rows, tagging each row's `Dataset` with `Aggregate` or `Synthetic`.
A selection is a list of `AttributeValue` terms; `describe_selection` prints it the way the
workflow does.

### Top attribute values, overall and for one city

In [ ]:
top_overall = top_attributes(
    result.synthetic,
    result.dp_aggregates,
    selection=[],
    num_values=10,
)
show(Markdown("**Top 10 attribute values across all complaints**"))
show(top_overall)

selection = [AttributeValue(attribute="city", value="Lakeside")]
print(describe_selection(selection))
top_lakeside = top_attributes(
    result.synthetic,
    result.dp_aggregates,
    selection=selection,
    show_attributes=["product_code", "age_range"],
    num_values=5,
)
show(Markdown("**Top 5 products and age ranges among Lakeside complaints**"))
show(top_lakeside)

if px is not None:
    px.bar(
        top_overall.to_pandas(),
        x="Count",
        y="Attribute Value",
        color="Attribute",
        orientation="h",
        title="Top attribute values (DP counts)",
        height=450,
    ).update_yaxes(autorange="reversed").show()

### Issues over time

`time_series` counts each value of the series attributes per value of the time attribute,
here the five issue flags per half-year `period`. Periods in which an issue never appears
are filled with zero so every line spans the full axis.

In [ ]:
issues = [
    "quality_issue",
    "price_issue",
    "service_issue",
    "delivery_issue",
    "description_issue",
]
series = time_series(
    result.synthetic,
    result.dp_aggregates,
    selection=[],
    time_attribute="period",
    series_attributes=issues,
)
show(Markdown(f"**{series.height} period x issue rows**"))
show(series)

if px is not None:
    px.line(
        series.to_pandas(),
        x="period",
        y="Count",
        color="Attribute Value",
        markers=True,
        title="Complaints per period by issue (DP counts)",
        height=450,
    ).show()

### Flows between cities and products

Links pair every source value with every target value and count the records that carry
both. `aggregate_links` reads the DP aggregates and is exact where the combination was
counted; `synthetic_links` counts synthetic rows and works for any depth. With a highlight
term each link also reports the `Proportion` of its records that carry that term.

`Anonymizer.links` chooses between the two: it uses aggregates when the pair plus the
selection plus the highlight fits within `reporting_length`, and synthetic rows otherwise.
Here `city` + `product_code` + `price_issue:True` is three terms, so the aggregate path is
taken and the two frames below agree exactly on `Count`.

In [ ]:
highlight = AttributeValue(attribute="price_issue", value="True")
links = Anonymizer.links(
    result,
    selection=[],
    source_attribute="city",
    target_attribute="product_code",
    highlight=highlight,
)
from_aggregates = aggregate_links(
    result.dp_aggregates, [], "city", "product_code", highlight
)
from_synthetic = synthetic_links(
    result.synthetic, [], "city", "product_code", highlight
)
assert links.equals(from_aggregates)

show(Markdown(f"**{links.height} city -> product links, share with a price issue**"))
show(links.sort("Count", descending=True).head(15))
show(
    Markdown(
        "**Same links from the synthetic rows, for comparison**"
        " (counts differ by the synthesis noise)"
    )
)
show(from_synthetic.sort("Count", descending=True).head(15))

if px is not None:
    import plotly.graph_objects as go

    nodes = links["Source"].unique(maintain_order=True).to_list()
    nodes += links["Target"].unique(maintain_order=True).to_list()
    index = {name: position for position, name in enumerate(nodes)}
    go.Figure(
        go.Sankey(
            node={"label": nodes},
            link={
                "source": [index[s] for s in links["Source"]],
                "target": [index[t] for t in links["Target"]],
                "value": links["Count"].to_list(),
                "label": [f"{p:.0%} price issue" for p in links["Proportion"]],
            },
        )
    ).update_layout(title="City to product flows (DP counts)", height=500).show()

## 6. Re-query without re-synthesizing, and tighten the budget

Every query above ran from `result` without touching pacsynth again. The same holds for a
deeper selection: two selection terms plus a source and target exceed the reporting
length, so `Anonymizer.links` switches to synthetic rows on its own.

Finally, a second run with `epsilon=4.0` shows the privacy-utility trade-off on the same
input: stronger privacy, more suppressed and fabricated long combinations. Because the
noise is random, compare the shape of the two reports rather than individual numbers.

In [ ]:
deep_selection = [
    AttributeValue(attribute="city", value="Lakeside"),
    AttributeValue(attribute="age_range", value="(30-40]"),
]
deep_links = Anonymizer.links(result, deep_selection, "product_code", "period")
print(describe_selection(deep_selection, multiline=False))
show(Markdown(f"**{deep_links.height} product -> period links for that selection**"))
show(deep_links.sort("Count", descending=True).head(10))

tighter = anonymizer.anonymize(sensitive, AnonymizeConfig(epsilon=4.0))
anonymizer.evaluate(tighter)
assert tighter.aggregate_error_report is not None
show(Markdown("**Aggregate error report at epsilon = 12**"))
show(result.aggregate_error_report.frame)
show(Markdown("**Aggregate error report at epsilon = 4**"))
show(tighter.aggregate_error_report.frame)

## Interpretation and limitations

- The released artifacts are `result.synthetic`, `result.aggregates_frame` and the two
  error reports. The error reports are computed against the exact counts, so they are
  themselves sensitive and are normally kept with the data owner, not published.
- Counts here are DP estimates of a synthetic example dataset. Differences between runs
  are expected; differences between the two error reports show the cost of sampling
  synthetic records on top of the DP aggregates.
- `epsilon` and `reporting_length` are the main levers. Longer reporting lengths answer
  deeper selections exactly but spread the budget over many more combinations.
- `sensitive_zeros` in `AnonymizeConfig` lists columns whose `"0"` values are real
  attributes; pacsynth otherwise treats `"0"` like a blank. This dataset has none.
- The package returns frames, not figures. The optional plotly cells show one way to draw
  them; the Streamlit workflow draws the same frames with its own colour schemes.